# Linear regression GPU baseline (PyTorch lstsq)

**Doc:** `torch.linalg.lstsq` on `[StandardScaler(X.fillna(0)) | 1]` is **exact OLS with intercept** on the scaled design — same as `sklearn` `Pipeline(StandardScaler, LinearRegression)` for that preprocessing. CPU: pandas, PCA, `GroupKFold`. GPU: tensor solve per fold.

Load `full_dataset_gpu.csv` / `genotypes_*_gpu.csv` when present. Saves `lr_gpu_weights.npy`, `lr_gpu_oof.csv`.

In [ ]:
# Cell 1 — imports & load
import json
import subprocess
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
sns.set_style("whitegrid")

_here = Path.cwd().resolve()
ROOT = _here if (_here / "maize_gxe_ml_pipeline.py").exists() else _here.parent
if not (ROOT / "maize_gxe_ml_pipeline.py").exists() and _here.name.lower() == "output":
    ROOT = _here.parent
sys.path.insert(0, str(ROOT))

from maize_gxe_ml_pipeline import (
    load_config,
    metrics_dict,
)

CFG_PATH = ROOT / "pipeline_config.yaml"
cfg = load_config(CFG_PATH)
out = Path(cfg["output_dir"])
if not out.is_absolute():
    out = (CFG_PATH.parent / out).resolve()

SUF = str(cfg.get("artifact_suffix", "_gpu"))
cohort = str(cfg["cohort"]).upper()
tr_min, tr_max = int(cfg["years"]["train_min"]), int(cfg["years"]["train_max"])
te_year = int(cfg["years"]["test_year"])
pca_n = int(cfg["hypers"]["pca_components"])
n_splits = int(cfg["hypers"]["group_cv_splits"])
rs = int(cfg.get("random_state", 42))
rng = np.random.default_rng(rs)

full_path = out / f"full_dataset{SUF}.csv"
if not full_path.exists():
    full_path = out / "full_dataset.csv"
geno_path = out / f"genotypes_{cohort.lower()}{SUF}.csv"
if not geno_path.exists():
    geno_path = out / f"genotypes_{cohort.lower()}.csv"

assert full_path.exists(), f"Missing {full_path} — run pipeline first."
assert geno_path.exists(), f"Missing {geno_path} — run pipeline first."

full_dataset = pd.read_csv(full_path, low_memory=False)
geno = pd.read_csv(geno_path, low_memory=False)
snp_cols = [c for c in geno.columns if c != "line_id"]

print("torch.cuda.is_available():", torch.cuda.is_available())
print(out, full_path.name, full_dataset.shape, geno_path.name, geno.shape)

In [ ]:
# Cell 2 — CPU scale + NaN handle; rebuild modeling frame (same as CPU notebook)
from maize_gxe_ml_pipeline import (
    fit_pca_for_lines,
    map_pca_to_rows,
    step5_environment_features,
    step6_select_env_and_gxe,
)

train_year_mask = full_dataset["YEAR"].between(tr_min, tr_max) & full_dataset["YLDBE"].notna()
test_year_mask = (full_dataset["YEAR"] == te_year) & full_dataset["YLDBE"].notna()
model_df = full_dataset.loc[train_year_mask | test_year_mask].copy()
model_df = model_df[model_df["line_id"].isin(set(geno["line_id"]))].copy()

max_rows = cfg.get("max_train_rows")
per_env = cfg.get("max_rows_per_env")
train_only = model_df.loc[model_df["YEAR"].between(tr_min, tr_max)]
if per_env:
    parts = []
    for env_id, chunk in train_only.groupby("env_id"):
        if len(chunk) > int(per_env):
            parts.append(chunk.sample(n=int(per_env), random_state=int(rng.integers(0, 1_000_000))))
        else:
            parts.append(chunk)
    train_only = pd.concat(parts, axis=0)
if max_rows is not None and len(train_only) > int(max_rows):
    train_only = train_only.sample(n=int(max_rows), random_state=42)
test_only = model_df.loc[model_df["YEAR"] == te_year]
model_df = pd.concat([train_only, test_only], axis=0).drop_duplicates()
snp_in_m = [c for c in snp_cols if c in model_df.columns]
if snp_in_m:
    model_df = model_df.drop(columns=snp_in_m, errors="ignore")

model_df, env_numeric, _ = step5_environment_features(model_df, str(cfg.get("env_scaling", "global")))
train_lines = model_df.loc[model_df["YEAR"].between(tr_min, tr_max), "line_id"].unique()
_, final_pca_pipe = fit_pca_for_lines(geno, snp_cols, train_lines, pca_n, rs)
gpc_df = map_pca_to_rows(geno, snp_cols, model_df["line_id"], final_pca_pipe)
gpc_cols = list(gpc_df.columns)
for c in gpc_cols:
    model_df[c] = gpc_df[c].values

is_train = model_df["YEAR"].between(tr_min, tr_max).values
env_only_numeric = [c for c in env_numeric if c in model_df.columns and not str(c).startswith("GPC_")]
engxe_path = out / f"env_gxe_feature_columns{SUF}.csv"
if not engxe_path.exists():
    engxe_path = out / "env_gxe_feature_columns.csv"
if engxe_path.exists():
    nongeno_feature_cols = pd.read_csv(engxe_path)["env_and_gxe_column"].astype(str).tolist()
    for c in nongeno_feature_cols:
        if c in model_df.columns:
            continue
        if c.startswith("GXE__") and "__x__" in c:
            rest = c[len("GXE__") :]
            gc, ec = rest.split("__x__", 1)
            model_df[c] = model_df[gc].astype(float) * model_df[ec].astype(float)
        else:
            raise KeyError(c)
else:
    past_cols = [c for c in ("past_yld_mean", "past_yld_median") if c in model_df.columns]
    env_chosen, pair_list = step6_select_env_and_gxe(
        model_df.loc[is_train].copy(), env_only_numeric, gpc_cols, rng,
        int(cfg.get("max_env_features_gxe", 20)), int(cfg.get("n_interaction_pairs", 10)),
    )
    interact_cols = []
    for gc, ec in pair_list:
        cname = f"GXE__{gc}__x__{ec}"
        model_df[cname] = model_df[gc].astype(float) * model_df[ec].astype(float)
        interact_cols.append(cname)
    nongeno_feature_cols = [c for c in env_chosen + past_cols + interact_cols if c in model_df.columns]

feature_cols = [c for c in gpc_cols + nongeno_feature_cols if c in model_df.columns]
model_df = model_df.reset_index(drop=True)
is_train = model_df["YEAR"].between(tr_min, tr_max).values
G = model_df[gpc_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
Ng = model_df[nongeno_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
X_all = np.hstack([G, Ng])
y_all = model_df["YLDBE"].values.astype(float)
groups_all = model_df["env_id"].astype(str).values
train_idx = is_train
print("features:", len(feature_cols), "rows:", len(model_df))

In [ ]:
# Cell 3 — GPU linear solve: GroupKFold OOF + final lstsq; nvidia-smi + timeit
def check_gpu_mem_mib():
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=15, check=False,
        )
        if r.returncode != 0 or not (r.stdout or "").strip():
            return None
        return float((r.stdout or "").strip().splitlines()[0])
    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError, IndexError):
        return None

def ols_lstsq_gpu(X_np, y_np, device):
    X_np = np.asarray(X_np, dtype=np.float64)
    y_np = np.asarray(y_np, dtype=np.float64).reshape(-1, 1)
    n = X_np.shape[0]
    X_t = torch.tensor(X_np, device=device, dtype=torch.float32)
    y_t = torch.tensor(y_np, device=device, dtype=torch.float32)
    ones = torch.ones((n, 1), device=device, dtype=torch.float32)
    Xa = torch.cat([X_t, ones], dim=1)
    sol = torch.linalg.lstsq(Xa, y_t).solution
    w = sol[:-1, 0].detach().cpu().numpy()
    b = float(sol[-1, 0].detach().cpu().item())
    return w, b

def predict_gpu(X_np, w, b, device):
    with torch.no_grad():
        Xt = torch.tensor(np.asarray(X_np, dtype=np.float32), device=device, dtype=torch.float32)
        wt = torch.tensor(w.astype(np.float32), device=device, dtype=torch.float32)
        return (Xt @ wt.reshape(-1) + b).cpu().numpy()

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("GPU unavailable; falling back to CPU sklearn-style lstsq (same math).")

df_train = model_df.loc[train_idx].reset_index(drop=True)
y_tr = y_all[train_idx]
g_tr = groups_all[train_idx]
gkf = GroupKFold(n_splits=n_splits)
oof_train = np.full(len(y_tr), np.nan)
fold_metrics = []

print("nvidia-smi pre-OOF:", check_gpu_mem_mib())
t0 = time.perf_counter()
for tr, te in gkf.split(df_train, y_tr, groups=g_tr):
    tr_lines = df_train.iloc[tr]["line_id"].unique()
    _, pipe = fit_pca_for_lines(geno, snp_cols, tr_lines, pca_n, rs)
    gpc_tr = map_pca_to_rows(geno, snp_cols, df_train.iloc[tr]["line_id"], pipe).values
    gpc_te = map_pca_to_rows(geno, snp_cols, df_train.iloc[te]["line_id"], pipe).values
    Xtr = np.nan_to_num(np.hstack([gpc_tr, df_train.iloc[tr][nongeno_feature_cols].values.astype(float)]), nan=0.0)
    Xte = np.nan_to_num(np.hstack([gpc_te, df_train.iloc[te][nongeno_feature_cols].values.astype(float)]), nan=0.0)
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)
    ytr = np.nan_to_num(y_tr[tr], nan=0.0)
    w, b = ols_lstsq_gpu(Xtr_s, ytr, device)
    oof_train[te] = predict_gpu(Xte_s, w, b, device)
    fold_metrics.append(metrics_dict(y_tr[te], oof_train[te]))
t_oof = time.perf_counter() - t0
print("nvidia-smi post-OOF:", check_gpu_mem_mib(), "| OOF wall:", round(t_oof, 3), "s")

oof_full = np.full(len(y_all), np.nan)
oof_full[np.where(train_idx)[0]] = oof_train
glob_oof = metrics_dict(y_tr, oof_train)
fold_rmses = [float(fm["rmse"]) for fm in fold_metrics if np.isfinite(fm["rmse"])]
cv_rmse_std = float(np.std(fold_rmses)) if fold_rmses else float("nan")

X_tr_all = np.nan_to_num(X_all[train_idx], nan=0.0)
y_tr_all = np.nan_to_num(y_all[train_idx], nan=0.0)
scaler_final = StandardScaler()
X_tr_s = scaler_final.fit_transform(X_tr_all)
t1 = time.perf_counter()
w_final, b_final = ols_lstsq_gpu(X_tr_s, y_tr_all, device)
t_fit = time.perf_counter() - t1
print("Final lstsq:", round(t_fit, 4), "s | OOF metrics:", glob_oof)

X_all_s = scaler_final.transform(np.nan_to_num(X_all, nan=0.0))
pred_all = predict_gpu(X_all_s, w_final, b_final, device)

np.save(out / "lr_gpu_weights.npy", np.concatenate([w_final.astype(np.float64), np.array([b_final])]))
joblib.dump(scaler_final, out / "lr_gpu_scaler.pkl")
lr_gpu_oof = model_df[["line_id", "env_id", "YEAR", "LOC", "YLDBE"]].copy()
lr_gpu_oof["oof_pred_lr_gpu"] = oof_full
lr_gpu_oof["pred_final_lr_gpu"] = pred_all
lr_gpu_oof.to_csv(out / "lr_gpu_oof.csv", index=False)

### 4. Model parameters (interpretable OLS on scaled design)

**GPU note:** Coefficients from `torch.linalg.lstsq` match **CPU `sklearn` LinearRegression** on the same `(StandardScaler, X)` — these visuals are identical.

**Reading coefs:** On the **standardized** design, a positive coefficient means **+1 SD** in that feature associates with **+coef** bushels/acre, holding others fixed (linear assumption). The intercept is **predicted yield when all scaled features are 0** (i.e., at each feature's training mean).

*Linear assumes independent additive linear effects; residual plots (below) test whether that is plausible for maize GxE.*

In [ ]:
# Cell 4 — model params (post-OOF fit): intercept, coef_df, bucket sums
X_cols = feature_cols
intercept = float(b_final)

coef_df = pd.DataFrame({"Feature": X_cols, "Coef": w_final})
coef_df["abs_coef"] = coef_df["Coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef")


def bucket(name: str) -> str:
    if name.startswith("GPC_"):
        return "Genetic"
    if name.startswith("GXE__"):
        return "GxE"
    if any(x in name.upper() for x in ("PRCP", "TAVG", "HTDD", "CLDD", "DP01", "DP10")):
        return "Weather"
    if any(x in name.upper() for x in ("CLAY", "SILT", "SAND", "PH", "SOC", "NITROGEN", "CFVO")):
        return "Soil"
    if "past_yld" in name:
        return "Past_pheno"
    return "Other"


coef_df["bucket"] = coef_df["Feature"].map(bucket)
bucket_sums = coef_df.groupby("bucket")["Coef"].sum().to_dict()

tr_df = model_df.loc[train_idx].copy()
y = tr_df["YLDBE"].values
oof_preds = oof_train

print(f"Intercept (baseline yield when scaled feats = 0 / at feature means): {intercept:.2f} bu/ac")
print("\nTop 10 Coefs (Δ yield per +1 SD of scaled feature, others fixed):")
print(coef_df.drop(columns="bucket").head(10).round(4))
print("\nBucket impacts (sum of coefs in scaled space):")
print(pd.Series(bucket_sums).sort_values(ascending=False).round(2))
print("\nPositive coef: +1 SD of that feature → +coef bu/ac (linear, holding others fixed).")

### 5. Actual vs predicted (OOF)

**Perfect fit** = points on the diagonal. **Spread** = prediction error. Gaps vs the diagonal motivate **non-linear** models (e.g. XGBoost GPU) when GxE is curved or interaction-heavy.

In [ ]:
# Cell 5 — actual vs predicted (OOF train rows)
plt.figure(figsize=(8, 8))
plt.scatter(y, oof_preds, alpha=0.6, s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--", lw=2)
plt.xlabel("Actual YLDBE (bu/ac)")
plt.ylabel("Predicted (OOF)")
plt.title("OOF: Actual vs Predicted Yield")
plt.savefig(out / "lr_actual_pred.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Residual diagnostics

**Residuals vs predicted:** random cloud is consistent with **homoscedastic** linear errors; funnels suggest **misspecification**. **Residuals by LOC** flag **env bias** (GxE). **Histogram** and **top coef** panel complement the global fit table.

In [ ]:
# Cell 6 — residuals: vs pred, by LOC, top coefs, histogram
residuals = y - oof_preds
full_df = pd.DataFrame(
    {
        "Pred": oof_preds,
        "Resid": residuals,
        "LOC": tr_df["LOC"].astype(str).values,
        "YEAR": tr_df["YEAR"].values,
        "Line": tr_df["line_id"].values,
    }
)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes[0, 0].scatter(full_df["Pred"], full_df["Resid"], alpha=0.6, s=10)
axes[0, 0].axhline(0, color="r", ls="--", lw=2)
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Residual")
axes[0, 0].set_title("Residuals vs Predicted (random scatter = good linear fit)")

if full_df["LOC"].nunique() > 30:
    top_locs = full_df["LOC"].value_counts().head(30).index
    sub_loc = full_df[full_df["LOC"].isin(top_locs)]
else:
    sub_loc = full_df
sns.boxplot(data=sub_loc, x="LOC", y="Resid", ax=axes[0, 1])
axes[0, 1].axhline(0, color="r", ls="--", lw=2)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha="right")
axes[0, 1].set_title("Residuals by LOC (pattern → env bias / GxE)")

top20_coef = coef_df.head(20).iloc[::-1]
sns.barplot(data=top20_coef, y="Feature", x="Coef", hue="bucket", dodge=False, ax=axes[1, 0])
axes[1, 0].axvline(0, color="gray", lw=0.8)
axes[1, 0].set_title("Top feature coefficients (absolute)")

sns.histplot(residuals, kde=True, ax=axes[1, 1])
axes[1, 1].axvline(0, color="r", ls="--", lw=2)
axes[1, 1].set_title("Residual distribution (normality check)")

plt.tight_layout()
plt.savefig(out / "lr_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

### 7. Maize-specific partial views & summary

**Pred vs key traits:** exploratory **partial** relationships (linear model combines all features; these are **bivariate** checks). **Q-Q plot:** if residuals deviate from the diagonal at tails, consider **YLDBE transform** or **non-linear** / **tree** models.

**Takeaway:** Residuals patterned by **LOC** → strengthen **GxE** / env features. **Non-normal** residuals → consider transforms or **XGBoost GPU** baseline comparison.

In [ ]:
# Cell 7 — maize scatterplots, Q-Q, summary table, CSV/JSON


def _pick_col(candidates):
    for c in candidates:
        if c in model_df.columns:
            return c
    return None


key_feats = [
    c
    for c in [
        _pick_col(["PHT", "PHT_BE", "pht"]),
        _pick_col(["past_yld_mean", "past_yld_median"]),
        _pick_col(["summer_tavg"]),
    ]
    if c is not None
]

plot_df = tr_df.copy()
plot_df["Pred"] = oof_preds
for feat in key_feats:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=plot_df, x=feat, y="Pred", alpha=0.6, s=12)
    plt.xlabel(feat)
    plt.ylabel("Predicted YLDBE (OOF)")
    plt.title(f"Pred yield vs {feat}")
    safe = str(feat).replace("/", "_").replace(" ", "_")
    plt.savefig(out / f"lr_vs_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title("Q-Q plot: residuals vs normal")
plt.savefig(out / "lr_qq.png", dpi=150, bbox_inches="tight")
plt.show()

summary = pd.DataFrame(
    {
        "Metric": ["R2 (OOF)", "RMSE (bu/ac)", "MAE", "Resid mean", "Resid std"],
        "Value": [
            r2_score(y, oof_preds),
            np.sqrt(mean_squared_error(y, oof_preds)),
            mean_absolute_error(y, oof_preds),
            float(np.mean(residuals)),
            float(np.std(residuals)),
        ],
    }
).round(3)
print("LinearRegression (GPU lstsq) summary:\n", summary.to_string(index=False))
summary.to_json(out / "lr_visual_summary.json", orient="records", indent=2)

coef_df.drop(columns="bucket", errors="ignore").to_csv(out / "lr_coefs.csv", index=False)
full_df.to_csv(out / "lr_interpretability_df.csv", index=False)

print("LR visuals added – interpretable baseline ready for hack demo!")

In [ ]:
# Cell 8 — metrics by env, GBR compare, residual/coef slides, rankings, lr_gpu_summary.json
display(pd.DataFrame([glob_oof]).T.rename(columns={0: "value"}))

by_env_rows = []
for eid, chunk in tr_df.groupby("env_id"):
    m = metrics_dict(chunk["YLDBE"].values, oof_full[chunk.index.values])
    m["env_id"] = eid
    m["n"] = len(chunk)
    by_env_rows.append(m)
metrics_by_env = pd.DataFrame(by_env_rows).sort_values("rmse", na_position="last")
metrics_by_env.to_csv(out / "lr_gpu_metrics_by_env_oof.csv", index=False)

summary_gpu = out / f"run_summary{SUF}.json"
if summary_gpu.exists():
    with open(summary_gpu, encoding="utf-8") as f:
        summ = json.load(f)
    go = summ.get("oof_global_metrics", {})
    display(
        pd.DataFrame(
            {
                "model": ["Tree pipeline OOF", "PyTorch lstsq OOF"],
                "rmse": [go.get("rmse"), glob_oof["rmse"]],
                "mae": [go.get("mae"), glob_oof["mae"]],
                "r2": [go.get("r2"), glob_oof["r2"]],
            }
        )
    )

fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(0, color="gray", lw=0.8)
ax.scatter(oof_train, tr_df["YLDBE"].values - oof_train, alpha=0.2, s=10, edgecolors="none")
ax.set_xlabel("OOF pred")
ax.set_ylabel("Residual")
ax.set_title("PyTorch lstsq OOF residuals")
fig.tight_layout()
fig.savefig(out / "lr_gpu_residuals.png", dpi=150)
plt.show()

feat_df = coef_df.rename(columns={"Feature": "feat", "Coef": "coef"})
feat_df["abs_coef"] = feat_df["coef"].abs()
top20 = feat_df.sort_values("abs_coef", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top20, y="feat", x="coef", hue="bucket", dodge=False, ax=ax)
ax.axvline(0, color="gray", lw=0.8)
ax.set_title("Top 20 |coef| (GPU lstsq, scaled design)")
fig.tight_layout()
fig.savefig(out / "lr_gpu_coef_plot.png", dpi=150, bbox_inches="tight")
plt.show()
display(feat_df.groupby("bucket")["abs_coef"].sum().reset_index())

topk = int(cfg.get("topk", 10))
te_mask = model_df["YEAR"].values == te_year
rank_rows = []
for env_id, chunk in model_df.loc[te_mask].groupby("env_id"):
    order = np.argsort(-pred_all[chunk.index.values])
    top_idx = chunk.index.values[order[:topk]]
    sub = model_df.loc[top_idx].copy()
    sub["pred_final_lr_gpu"] = pred_all[top_idx]
    sub["rank"] = np.arange(1, len(sub) + 1)
    sub["uncertainty_oof_rmse"] = glob_oof["rmse"]
    rank_rows.append(sub)
lr_rank = pd.concat(rank_rows, axis=0) if rank_rows else pd.DataFrame()
lr_rank.to_csv(out / "lr_gpu_rankings_topk_per_env.csv", index=False)

interp_pngs = [
    str(out / "lr_actual_pred.png"),
    str(out / "lr_diagnostics.png"),
    str(out / "lr_qq.png"),
] + sorted(str(p) for p in out.glob("lr_vs_*.png"))

lr_summary = {
    "model": "PyTorch lstsq OLS (GPU if CUDA)",
    "device": device,
    "oof_global_metrics": glob_oof,
    "cv_rmse_std": cv_rmse_std,
    "wall_seconds": {"oof_loop": float(t_oof), "final_lstsq": float(t_fit)},
    "outputs": [str(out / "lr_gpu_weights.npy"), str(out / "lr_gpu_oof.csv")],
    "interpretability_pngs": interp_pngs,
    "interpretability_tables": [str(out / "lr_coefs.csv"), str(out / "lr_visual_summary.json")],
}


def _jd(o):
    return o.item() if isinstance(o, (np.floating, np.integer)) else str(o)


with open(out / "lr_gpu_summary.json", "w", encoding="utf-8") as f:
    json.dump(lr_summary, f, indent=2, default=_jd)

print("GPU pipelines ready – run nvidia-smi during fit to confirm.")